## UX Critic Agent

Organizing documents and utilizing Chroma DB\
Retrieving documents for RAG pipeline\
Test with sample documents\
Define structured format for inputs and outputs\
Prompt and LLM chain for the critique of the storyboard\

Input documents and use the documents to create a RAG pipeline that will critique the storyboard generated specifically on its painpoints

In [34]:
# Core imports
import json
import re
import sys
from pathlib import Path
from typing import List, Dict, Optional, Literal

# Add inclass to path for llm_utils
sys.path.insert(0, str(Path("../../inclass").resolve()))

# LangChain imports
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document

# ChromaDB
import chromadb

# Pydantic for structured output
from pydantic import BaseModel, Field

# NumPy for vector operations
import numpy as np

# BM25 for keyword-based search
from rank_bm25 import BM25Okapi

# Our utilities
from llm_utils import get_llm, get_embeddings, get_ollama_client, get_chat_model

# imports from other agents
from models.schemas import Panel, StoryboardOutput

In [43]:
# Configuration
USE_REMOTE = False  # Set True to use remote server

# Initialize LLM with llama3.2:3b (default)
llm = get_llm(use_remote=USE_REMOTE, model="3b", temperature=0.3)
chat_model = get_chat_model(use_remote=USE_REMOTE, model="3b")
parser = StrOutputParser()

# Initialize Ollama client for embeddings
ollama_client = get_ollama_client()

# Test connection
print("\nTesting LLM connection...")
response = llm.invoke("Say 'Documentation Assistant ready!' in 5 words or less.")
print(f"Response: {response}")

Using Ollama at http://localhost:11434
Model: llama3.2:3b
Using Ollama at http://localhost:11434
Model: llama3.2:3b
Ollama client connected to: http://localhost:11434

Testing LLM connection...
Response: "Documentation Assistant ready!"


### Loading Documents

In [36]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# load and chunk pdf documents
def load_pdfs(pdf_dir: str):
    """load and chunk PDFs from a folder"""
    loader = DirectoryLoader(
        pdf_dir,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader
    )
    raw_docs = loader.load()

    # Split into chunks — important for retrieval accuracy
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,       # ~a paragraph
        chunk_overlap=50,     # overlap so context isn't cut off
    )
    chunks = splitter.split_documents(raw_docs)
    print(f"Loaded {len(raw_docs)} pages → {len(chunks)} chunks")
    return chunks

In [5]:
# call pdfs and load into ChromaDB
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

# initialize ChromaDB client
CHROMA_PATH = "./data/chroma_db"
COLLECTION = "ux-research-docs"

# load and chunk your PDFs
chunks = load_pdfs("UX-documents")

# get embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")


# build vector store with ChromaDB
vector_store = Chroma(
    collection_name = COLLECTION,
    embedding_function = embeddings,
    persist_directory = CHROMA_PATH
)

vector_store.add_documents(chunks)



Loaded 65 pages → 227 chunks


['23a4be49-d8af-4ac6-a4c8-7b8defc3dfdf',
 '5e44fff6-20d9-427f-abc5-43a38afe75ac',
 '2ff61df2-ddc7-4dce-9f0d-8c1b2c11cb90',
 'ce2caddb-d31c-49af-8801-c794a12a6be8',
 '69ff0ed2-4c94-44f4-8a8a-1ae2a2e0b907',
 '28dad505-645a-42f2-8fb4-edbb5f89d348',
 '2ce5bfca-1f4e-4cec-81fc-a1edcf413050',
 'd50ada0b-77c1-4556-8373-431ec8608626',
 '42f0f3d6-e627-45d8-92e6-94a306697e7a',
 'c40f84cf-7c9c-4211-8341-f6fb28459b4b',
 '76281598-85b2-43ba-94d6-56f96b33f101',
 '689a4983-69ce-48e8-9649-78b253b26d59',
 '10156c2c-7892-4dc2-aa28-22f08a196353',
 'bdb22265-b39c-4102-bab3-364a8e9e6be1',
 '2eb7bc21-acae-4ca6-ab40-67b419924076',
 '7ed09c36-6041-4c95-833f-9a91579e81c7',
 'ce9746b5-cac7-4c61-8032-99389418edf7',
 '14f93c1f-17d7-43e8-bfe2-54fcdd6c8ae3',
 '70ae4119-0bb1-459e-8129-7fe9ef7295ed',
 '7352743e-844b-4e81-8b43-2a9cf772b27d',
 'd9aece9c-911a-4a7c-8668-2b0a4eb24935',
 '7790eeb8-ec39-4823-9f42-5c30edb65f2b',
 'bd1d6fcf-724f-471f-b6da-78dea77268b6',
 '9b71af91-cd34-4951-b96f-62b0134f2c2b',
 '4b80705d-37e0-

### Pydantic Models

In [46]:
# create critic input for UX Critic Agent
class CriticInput(BaseModel):
    """Input to the Critic Agent"""
    panels: List[Panel] = Field(..., description="List of UI panels to critique")
    retrieved_docs: List[str] = Field(..., description="Relevant documentation snippets retrieved from the vector store")

In [ ]:
# create critic output panel for UX Critic Agent
class PanelCritique(BaseModel):
    """Panel output from the Critic Agent"""
    panel_number: int = Field(..., description="Index of the panel being critiqued")
    pain_point: str = Field(..., description="Description of the identified pain point")
    reason: str = Field(..., description="Explanation of why this is a pain point based on the retrieved documentation")
    severity: Literal["Low", "Medium", "High"] = Field(..., description="Severity of the pain point based on the documentation and potential user impact")

In [48]:
# create critic output for UX Critic Agent
class CriticOutput(BaseModel):
    """Output from the Critic Agent"""
    critiques: List[PanelCritique] = Field(..., description="List of critiques for each panel")

### Retrieve documents

In [27]:
def basic_retrieve(panels: List[Panel], top_k: int) -> List[str]:
    """Retrieve the top k most similar documents for a query.

    Args:
        panels: List of UI panels to search for
        top_k: Number of documents to retrieve

    Returns:
        List of dictionaries with 'content', 'metadata', and 'score' keys"""
    # initialize retriever
    retriever = vector_store.as_retriever(search_kwargs={"k": top_k})
    # combine panel content into a single query
    query = " ".join([f"{panel.action} {panel.context}" for panel in panels])
    # retrieve top 5 documents
    docs = retriever.invoke(query)
    return [doc.page_content for doc in docs]


    

In [29]:
# load in files
with open('sample_panel_outputs.json', "r") as f:
    SAMPLE_PANEL_OUTPUTS = json.load(f)

# test retrieval with sample panel output
test_input = SAMPLE_PANEL_OUTPUTS[0]
# convert dicts to panel objects
panels = [Panel(**p) for p in test_input["panels"]]
retrieved = basic_retrieve(panels, 5)

print(f"Retrieved {len(retrieved)} chunks\n")
for i, doc in enumerate(retrieved):
    print(f"[Doc {i+1}]")
    print(doc[:200])
    print("---")

Retrieved 5 chunks

[Doc 1]
Conference.  Upon  calling  customer  support  to  reschedule,  I  was  given  the  
choice
 
to
 
take
 
the
 
assigned
 
appointment
 
or
 
reschedule
 
my
 
delivery
 
appointment
 
to
 
another
 

---
[Doc 2]
for
 
25%
 
off
 
your
 
next
 
order,
 
but
 
when
 
you
 
try
 
to
 
enter
 
it,
 
the
 
code
 
isn’t
 
accepted.
  
That’s
 
a
 
blocker
 
for
 
the
 
user,
 
and
 
may
 
result
 
in
 
a
 
lost
 
c
---
[Doc 3]
product
 
on
 
a
 
website.
  
It
 
includes
 
the
 
scenario
 
and
 
expectations
 
of
 
the
 
user,
 
steps
 
they
 
would
 
take,
 
and,
 
importantly,
 
what
 
they
 
are
 
thinking
 
and
 
feelin
---
[Doc 4]
Let’s  imagine  the  website  of  a  catering  company.  Allergy  considerations  now  play  a  
huge
 
part
 
in
 
food
 
selection
 
for
 
large
 
events.
 
A
 
website
 
where
 
allergy
 
informati
---
[Doc 5]
One  often-forgotten  friction  point  is  account  linking  and  verification.  Slow  and  
buggy
 
processes
 
leave
 

### Basic RAG Chain

In [37]:
# RAG prompt template for Critic Agent

critic_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a UX Critic Agent. Your job is to analyze the provided storyboard panels and
identify potential UX pain points based on the retrieved documentation.
For each panel, determine if there are any UX issues that could arise based on the user's action and context.
Use the retrieved documentation and context to support your analysis and provide a severity rating for each identified pain point.
Return only valid JSON, no extra text."""),
    ("human", """Here are the storyboard panels:
{panels}

Here is the relevant UX research:
{retrieved_docs}

Identify the UX pain points in these panels backed by the research above.
Format your response as JSON with this structure:
{{
    "critiques": [
        {{
        "panel": <index of the panel being critiqued>,
        "pain_point": <description of the identified pain point>,
        "reason": <explanation of why this is a pain point based on the retrieved documentation>,
        "severity": <severity of the pain point based on the documentation and potential user impact (Low, Medium, High)>
        }}
     ]
}}""")
])

# format panels for RAG chain
def format_panels(panels: List[Panel]) -> str:
    return "\n".join([
        f"Panel {p.panel_number}: Action='{p.action}', Context='{p.context}', Emotion='{p.emotion}'"
        for p in panels
    ])

# format retrieved docs for RAG chain
def format_retrieved_docs(retrieved_docs: List[str]) -> str:
    return "\n\n".join([f"[Doc {i+1}]: {doc}" for i, doc in enumerate(retrieved_docs)])



In [38]:
# function to run basic rag
def basic_rag(panels: List[Panel], top_k: int):
    # retrieve relevant docs
    retrieved_docs = basic_retrieve(panels, top_k)

    # format inputs for prompt
    formatted_panels = format_panels(panels)
    formatted_docs = format_retrieved_docs(retrieved_docs)

    # rag chain
    rag_chain = critic_prompt | chat_model | parser
    # run prompt
    response = rag_chain.invoke({
        "panels": formatted_panels,
        "retrieved_docs": formatted_docs
    })

    return response

In [39]:
# test RAG chain with sample panels
test_input = SAMPLE_PANEL_OUTPUTS[0]
panels = [Panel(**p) for p in test_input["panels"]]
response = basic_rag(panels, 5)
print("RAG Chain Response:")
print(response)

RAG Chain Response:
{
    "critiques": [
        {
            "panel": 2,
            "pain_point": "Search results loading slowly delays flight selection, increasing user stress",
            "reason": "Doc 2 identifies 'lengthy user journeys' and 'non-intuitive flows' as common pain points that cause user stress during critical tasks like flight selection",
            "severity": "Medium"
        },
        {
            "panel": 3,
            "pain_point": "Ineffective filtering results in overwhelming irrelevant options, causing frustration",
            "reason": "Doc 2 highlights 'confusing navigation' and 'complex user journeys' as pain points that lead to user frustration when filtering options",
            "severity": "Medium"
        },
        {
            "panel": 4,
            "pain_point": "Mandatory account creation before checkout creates unnecessary friction",
            "reason": "Doc 5 explicitly states that slow and buggy account verification processes signif

In [53]:
# test RAG chain with sample panels
test_input = SAMPLE_PANEL_OUTPUTS[1]
panels = [Panel(**p) for p in test_input["panels"]]
response = basic_rag(panels, 5)
print("RAG Chain Response:")
parsed = json.loads(response)
for c in parsed["critiques"]:
    print(f"Panel Number: {c["panel"]}")
    print(f"Pain Point: {c["pain_point"]}")
    print(f"Reason: {c["reason"]}")
    print(f"Severity: {c["severity"]}")
    print("-" * 40)

RAG Chain Response:
Panel Number: 1
Pain Point: Uncertainty about security when opening job listing app
Reason: The context of feeling uncertain about security when linking accounts or making first transactions suggests that users may feel vulnerable when accessing sensitive information.
Severity: Low
----------------------------------------
Panel Number: 2
Pain Point: Confusion from lack of filter options for hours and distance
Reason: The research highlights the importance of guiding users with clear, real-time explanations during critical moments, which could be hindered by the absence of filter options.
Severity: Medium
----------------------------------------
Panel Number: 3
Pain Point: Frustration from dense list with unclear labels
Reason: The research emphasizes the need for human-like reassurance and clear explanations, which could be compromised by a cluttered and confusing interface.
Severity: High
----------------------------------------
Panel Number: 4
Pain Point: Impatien